# 003 - Level3

## 3. Aggregations

### 📐 Operaciones de agregación

- **SUM()**  
  Calcula el **total de valores** (suma).

- **AVG()**  
  Calcula el **valor promedio** (media aritmética).

- **MIN() / MAX()**  
  Encuentra el **valor mínimo** o **máximo**.

- **GROUP BY**  
  Agrupa los datos y permite calcular **métricas por categoría**.

- **HAVING**  
  Filtra los resultados **después** de aplicar funciones de agregación.


In [1]:
import pandas as pd
import numpy as np
import polars as pl

In [2]:
df_matches = pd.read_csv('../Data/WorldCupMatches.csv').rename(columns=lambda col: col.lower())
df_players = pd.read_csv('../Data/WorldCupPlayers.csv').rename(columns=lambda col: col.lower())
df_cups = pd.read_csv('../Data/WorldCups.csv').rename(columns=lambda col: col.lower())

pl_matches = pl.read_csv('../Data/WorldCupMatches.csv').rename(lambda col: col.lower())
pl_players = pl.read_csv('../Data/WorldCupPlayers.csv').rename(lambda col: col.lower())
pl_cups = pl.read_csv('../Data/WorldCups.csv').rename(lambda col: col.lower())

## Ejercicio 3.1: El Dominio de los Goles Locales

### 🎯 Objetivo

Identificar qué selecciones han sido **más potentes históricamente cuando juegan como locales**.  
Para ello, se debe calcular el **total de goles anotados en casa** por cada equipo (`Home Team Name`) a lo largo de todos los mundiales.

---

### 📌 Condiciones del ejercicio

- Agrupar por el **nombre del equipo local** (`Home Team Name`).
- Sumar los **goles marcados en casa** (`Home Team Goals`).
- Filtrar los resultados para mostrar **solo equipos con más de 100 goles totales**.
- Ordenar los resultados de **mayor a menor** según el total de goles.

---

### 📊 Columnas a mostrar

- `Home Team Name`
- `Total_Goles_Local` (nombre sugerido para la agregación)

---

## 1. Postgres (SQL)

En SQL, la clave es recordar que:

- **`GROUP BY`** define qué columnas se pueden mostrar.
- **`HAVING`** permite filtrar sobre **resultados agregados**.

**Pregunta:**  
¿Cómo escribirías la consulta en **Postgres** para resolver este reto?

---

### 💡 Pista para lo que viene después

- **Pandas 🐼**  
  ```python
  .groupby('col').agg({'val': 'sum'})

- **Polars 🧊**  
```python
  .group_by('col').agg(pl.col('val').sum())


```SQL
SELECT
    "Home Team Name",
    SUM("Home Team Goals") AS total_goles_local
FROM worldcup.matches
GROUP BY "Home Team Name"
HAVING SUM("Home Team Goals") > 100
ORDER BY total_goles_local DESC;
```

In [15]:
df_m = df_matches.copy()

# Opcion 1
# # 1. Agrupamos y sumamos
# res_f = df_m.groupby('home team name')['home team goals'].sum().reset_index()

# # 2. Renombramos la columna para que se vea bien
# res_f.columns = ['home team name', 'total_goles_local']

# # 3. Ahora que es un DataFrame normal, filtramos (HAVING)
# res_f = res_f[res_f['total_goles_local'] > 100]

# # 4. Ordenamos (ORDER BY)
# res_f = res_f.sort_values(by='total_goles_local', ascending=False)

# Opción 2

res_f = df_m.groupby('home team name').agg(
    total_goles_local=('home team goals', 'sum')
).reset_index()

res_f = res_f[res_f['total_goles_local'] > 100].sort_values(by='total_goles_local', ascending=False)

res_f

,home team name,total_goles_local
7,Brazil,180.0
2,Argentina,111.0


In [19]:
res_f = pl_matches.group_by('home team name').agg(
    (pl.col('home team goals').sum().alias('total_goles_local'))
).filter(
    pl.col('total_goles_local') > 100
).sort(
    'total_goles_local', descending=True
)

res_f

home team name,total_goles_local
str,i64
"""Brazil""",180
"""Argentina""",111


## Ejercicio 3.2: Análisis de Sedes y Asistencia

### 🎯 Objetivo

Generar un **reporte estadístico por ciudad** para entender qué sedes han sido las **más activas** y cuál es el **volumen de público** que mueven.

---

### 📌 Condiciones del ejercicio

- Agrupar la información por la columna `City`.
- Contar el **número total de partidos** realizados en cada ciudad.
- Calcular el **promedio de asistencia** (`Attendance`) por ciudad.
- Ordenar los resultados para mostrar **primero las ciudades con mayor número de partidos**.

---

### 📊 Columnas a mostrar

- `City`
- `total_partidos` (conteo)
- `promedio_asistencia` (promedio)

---

## 1. Postgres (SQL)

En SQL, cuando necesitas **más de una métrica**, simplemente las separas con comas en el `SELECT`.  
Recuerda que para el promedio se utiliza **`AVG()`**.

**Pregunta:**  
¿Cómo escribirías la consulta en **Postgres** para obtener este reporte?

---

### 💡 Tip Pro

En un escenario real, la columna `Attendance` podría:

- Contener **valores nulos**
- Estar almacenada como **texto**

Por ahora, asumimos que es **numérica**, pero es importante tener en cuenta lo aprendido en el **Nivel 2 sobre limpieza de datos**.


```SQL
SELECT
    city,
    COUNT(*) AS total_partidos,
    AVG(attendance::integer) AS promedio_asistencia
FROM worldcup.matches
WHERE city IS NOT NULL
GROUP BY city
ORDER BY total_partidos DESC;
```

In [24]:
df_m = df_matches.copy()

df_m['attendance'] = pd.to_numeric(df_m['attendance'], errors='coerce')

res = df_m.groupby('city').agg(
    total_partidos = ('city','count'),
    promedio_asistenca = ('attendance', 'mean')
).reset_index()

res_f = res.sort_values(by='total_partidos', ascending=False)

In [31]:
res_f = pl_matches.filter(
    pl.col('city').is_not_null()
).group_by('city').agg([
    pl.len().alias('total_partidos'),
    pl.col('attendance').cast(pl.Int64, strict=False).mean().alias('promedio_asistencia')
]).sort('total_partidos', descending=True)

## Ejercicio 3.3: El Análisis de los Mundiales por Año

### 🎯 Objetivo

Analizar la **evolución histórica de los mundiales**.  
Para cada año (`Year`), se deben calcular las siguientes métricas:

- **Total de goles** anotados en ese mundial  
  (suma de `Home Team Goals` y `Away Team Goals`)
- **Máxima asistencia** (`Attendance`) registrada en un solo partido ese año
- **Promedio de goles por partido** en ese año

---

### 📌 Condiciones del ejercicio

- Eliminar las filas donde el valor de `Year` sea **nulo**.
- Asegurarse de que la columna `Attendance` sea tratada como **numérica**.
- Ordenar los resultados de forma **cronológica**  
  (del año más antiguo al más reciente).

---

### 📊 Columnas resultantes sugeridas

- `Year`
- `total_goles`
- `max_asistencia`
- `promedio_goles`

---

## 1. Postgres (SQL)

En SQL, es posible sumar dos columnas dentro de una función de agregación de esta forma:

```sql
SUM(columna1 + columna2)


```SQL
SELECT
    year,
    SUM("Away Team Goals"+"Home Team Goals") AS total_goles,
    MAX(attendance::INTEGER) AS max_asistencia,
    ROUND(AVG("Away Team Goals"+"Home Team Goals"),2) AS promedio_goles
FROM worldcup.matches
WHERE year IS NOT NULL
GROUP BY year
ORDER BY year DESC;
```

In [47]:
df_m = df_matches.copy()

res = df_m[(df_m['year'].notna())].copy()

res['total_goles_partido'] = res['home team goals'] + res['away team goals']

res['attendance'] = pd.to_numeric(res['attendance'], errors='coerce')

res_f = res.groupby('year').agg(
    total_goles = ('total_goles_partido', 'sum'),
    max_asistencia = ('attendance','max'),
    promedio_goles = ('total_goles_partido','mean')
).reset_index()

res_f['promedio_goles'] = res_f['promedio_goles'].round(2)

res_f = res_f.sort_values(by='year', ascending=False)

In [57]:
res_f = pl_matches.filter(
    pl.col('year').is_not_null()
).with_columns(
    total_goles_partido = pl.col('home team goals') + pl.col('away team goals')
).group_by('year').agg([
    pl.col('total_goles_partido').sum().alias('total_goles'),
    pl.col('attendance').cast(pl.Int64, strict=False).max().alias('max_asistencia'),
    pl.col('total_goles_partido').mean().round(2).alias('promedio_goles')
]).select(
    'year',
    'total_goles',
    'max_asistencia',
    'promedio_goles'
).sort('year', descending=True)

res_f

year,total_goles,max_asistencia,promedio_goles
i64,i64,i64,f64
2014,206,74738,2.58
2010,145,84490,2.27
2006,147,72000,2.3
2002,161,69029,2.52
1998,171,80000,2.67
…,…,…,…
1954,140,62500,5.38
1950,88,173850,4.0
1938,84,58455,4.67
